# M2 · Autograd

**Outcome:** Inspect the autograd lifecycle and control where gradients stop.

Run cells with **Shift + Enter**. PyTorch is already installed in standard Colab runtimes.

## Experiment question

> Which leaves will receive gradients, and which route will detach remove?

Before running code, write a prediction. Then observe the evidence, change one variable, and explain the difference.

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1 · Record a forward trail
Only the values that may learn need `requires_grad=True`. Inspect `is_leaf` and `grad_fn` before calling backward.

In [ ]:
x = torch.tensor(2.)
w = torch.tensor(-1., requires_grad=True)
b = torch.tensor(.5, requires_grad=True)
loss = (x*w + b)**2
print('w is leaf:', w.is_leaf, 'w.grad:', w.grad)
print('loss is leaf:', loss.is_leaf, 'loss.grad_fn:', loss.grad_fn)

## 2 · Walk backward and clear the answers
The graph is recorded during forward; `.backward()` fills gradients on the leaves.

In [ ]:
loss.backward()
print('loss:', loss.item(), 'dw:', w.grad.item(), 'db:', b.grad.item())
w.grad.zero_(); b.grad.zero_()
print('after clearing:', w.grad.item(), b.grad.item())

## 3 · Detach one branch
The two modes produce the same forward values. Compare which parameters receive gradients.

In [ ]:
x = torch.tensor(3.)
teacher_w = torch.tensor(2., requires_grad=True)
student_w = torch.tensor(1., requires_grad=True)
teacher_prediction = teacher_w * x
target = teacher_prediction.detach()
student_prediction = student_w * x
loss = (student_prediction - target).pow(2)
loss.backward()
print('target:', target.item(), 'loss:', loss.item())
print('student grad:', student_w.grad.item())
print('teacher grad:', teacher_w.grad)  # None: detach cut this route

## 4 · Control flow records the path that ran
Eager autograd builds a backward route from the chosen branch. A new call may record a different route.

In [ ]:
def branch_loss(x):
    if x.sum() > 0:
        y = torch.sin(x)
    else:
        y = torch.cos(x)
    return y.square().mean()

for values in [torch.ones(4), -torch.ones(4)]:
    x = values.requires_grad_()
    loss = branch_loss(x)
    loss.backward()
    print('sum:', x.sum().item(), 'loss grad_fn:', type(loss.grad_fn).__name__, 'grad:', x.grad)

## 5 · Compilation adds FX regions, guards, and possible graph breaks
A Python `if` on tensor data needs a runtime boolean and normally breaks Dynamo capture. Use `TORCH_LOGS="graph_breaks,recompiles,guards"` in a script to inspect boundaries and cache misses. `torch.cond` can represent restricted tensor-dependent branches as structured control flow.

In [ ]:
@torch.compile
def compiled_branch(x):
    if x.sum() > 0:       # tensor-data-dependent Python branch
        return torch.sin(x)
    return torch.cos(x)

print(compiled_branch(torch.ones(4)))
print(compiled_branch(-torch.ones(4)))

## 6 · Inspect two corner cases
A fixed Python loop is normally unrolled and guarded on its trip count, so a large loop can create a large graph. Structured branches must return compatible tensor metadata.

In [ ]:
def fixed_loop(x, steps: int):
    for _ in range(steps):
        x = torch.sin(x) + x
    return x

report = torch._dynamo.explain(fixed_loop)(torch.ones(4), 3)
print('captured graphs:', report.graph_count)
print('FX nodes:', len(list(report.graphs[0].graph.nodes)))
print(report.graphs[0].graph)

In [ ]:
def positive(x): return torch.sin(x)
def negative(x): return torch.cos(x)

x = torch.tensor([-1., 2.], requires_grad=True)
y = torch.cond(x.sum() > 0, positive, negative, (x,))
y.sum().backward()
print('selected output:', y.detach(), 'gradient:', x.grad)

A tensor-driven Python `while` has no compile-time trip count. PyTorch's prototype structured `while_loop` can preserve such a loop for compilation/export, but current documentation describes it as inference-only; do not treat it as an autograd training-loop replacement.

## 7 · In-place mutation is checked, not the defining feature
Autograd saves version counters with tensors needed for backward and raises rather than silently differentiating through a changed value.

In [ ]:
x = torch.ones(3, requires_grad=True)
y = x.exp()
y.add_(1)
try:
    y.sum().backward()
except RuntimeError as error:
    print(type(error).__name__, str(error).splitlines()[0])

## Try it
Remove `.detach()`, rebuild the tensors, and run again. Predict the teacher gradient before printing it. Then wrap both model calls in `torch.no_grad()` and inspect `requires_grad` on their outputs.

## Reflection

1. What did you predict?
2. What evidence did the output provide?
3. Which one variable did you change?
4. How does the result connect to the lesson's mental model?